# Fashion Image Search: Visual Product Discovery

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SerendipityOneInc/APIClaw-Notebook/blob/main/e-commerce-search/image_search.ipynb)

This notebook demonstrates how to use APIClaw's **Fashion Image Search** API to find visually similar fashion products from a catalog of 200M+ items. Upload any fashion image and discover matching products across major retailers.

### What you'll learn

1. **Basic image search** — find products visually similar to a query image
2. **Fashion item detection** — automatically locate bags, shoes, clothing, and accessories in any photo
3. **Image embedding generation** — extract feature vectors for detected items (for custom similarity search)
4. **Bounding box cropping** — focus on a specific item in a multi-item image
5. **Text-guided image search** — combine image + text for better matching
6. **Filtered image search** — apply brand, price, and retailer filters
7. **"Shop the Look" pipeline** — detect → embed → search for every item in a street-style photo

### API Reference

| Endpoint | Method | Description | Cost |
|----------|--------|-------------|------|
| `/openapi/v2/model/fashion-image-search` | POST | Search fashion products by image similarity | 1 credit |
| `/openapi/v2/model/fashion-product-search` | POST | Search fashion products by text | 1 credit |
| `/openapi/v2/model/image-detection` | POST | Detect fashion items → bounding boxes + category + confidence | 1 credit |
| `/openapi/v2/model/image-embedding` | POST | Detect items + generate feature embeddings (with optional tags) | 1 credit |
| `/openapi/v2/model/fashion-image-embedding` | POST | SigLIP2 768-dim image embeddings (batch up to 8 images) | 1 credit |
| `/openapi/v2/model/fashion-text-embedding` | POST | SigLIP2 768-dim text embeddings (batch up to 32 queries) | 1 credit |
| `/openapi/v2/model/fashion-similarity` | POST | Text-image cosine similarity scoring matrix | 1 credit |

## Setup

Sign up at [apiclaw.io/register](https://apiclaw.io/register) to get your API key — every new account includes 1,000 free credits to get started. Need more? Check out our [credit packages](https://apiclaw.io/pricing) for higher volumes.

In [ ]:
!pip install -q httpx Pillow matplotlib numpy

In [ ]:
import getpass
import os

if "APICLAW_API_KEY" not in os.environ:
    os.environ["APICLAW_API_KEY"] = getpass.getpass("Enter your APIClaw API key (hms_live_...): ")

API_KEY = os.environ["APICLAW_API_KEY"]
BASE_URL = "https://api.apiclaw.io/openapi/v2"

In [ ]:
import httpx
import json
import numpy as np
from PIL import Image, ImageDraw
from io import BytesIO
import matplotlib.pyplot as plt

client = httpx.Client(
    base_url=BASE_URL,
    headers={"Authorization": f"Bearer {API_KEY}"},
    timeout=30.0,
)

# ── Fashion category class IDs ──────────────────────────────────────────
FASHION_CLASSES = {
    0: "Bag", 1: "Cap", 2: "Down-Clothing", 3: "Glasses", 4: "Jewelry",
    5: "Others", 6: "Shoes", 7: "Sock", 8: "Up-Clothing", 9: "Watch",
}


# ── API wrapper functions ────────────────────────────────────────────────

def image_search(image_url: str, limit: int = 10, **kwargs) -> dict:
    """Search fashion products by image similarity."""
    payload = {"imageUrl": image_url, "limit": limit, **kwargs}
    resp = client.post("/model/fashion-image-search", json=payload)
    resp.raise_for_status()
    return resp.json()["data"]


def text_search(query: str, top_k: int = 5) -> list[dict]:
    """Search fashion products by text (helper for getting demo images)."""
    resp = client.post("/model/fashion-product-search", json={"query": query, "topK": top_k})
    resp.raise_for_status()
    return resp.json()["data"]["products"]


def detect_fashion_items(image_url: str, top_k: int = 10, classes: list[int] | None = None,
                         return_image: bool = False) -> dict:
    """Detect fashion items in an image → bounding boxes + category + confidence.

    Uses: POST /model/image-detection

    Args:
        image_url: HTTPS URL of the image to analyze.
        top_k: Max number of detections to return.
        classes: Filter by category IDs (e.g. [0, 6] for Bag + Shoes). None = all categories.
        return_image: If True, returns a base64 annotated image with bounding boxes drawn.

    Returns:
        dict with 'detections' list, each containing:
            - boundingBox: [x1, y1, x2, y2]
            - classId: int (0=Bag, 1=Cap, ..., 9=Watch)
            - score: float (detection confidence 0-1)
            - areaRatio: float (object area / image area)
            - centerDistanceRatio: float (distance from image center)
    """
    payload = {"image": image_url, "topK": top_k}
    if classes is not None:
        payload["classes"] = classes
    if return_image:
        payload["returnImage"] = True
    resp = client.post("/model/image-detection", json=payload)
    resp.raise_for_status()
    return resp.json()["data"]


def get_image_embedding(image_url: str, with_tag: bool = True, with_embedding: bool = True,
                        bounding_boxes: list[list[int]] | None = None,
                        text: list[str] | None = None) -> dict:
    """Detect fashion items and generate feature embeddings for each.

    Uses: POST /model/image-embedding

    This is the all-in-one endpoint: it detects items, classifies them,
    and generates embeddings — all in a single API call.

    Args:
        image_url: HTTPS URL of the image.
        with_tag: Include fashion category tags (e.g. "Bag", "Shoes") with confidence.
        with_embedding: Include feature embedding vectors for each detected item.
        bounding_boxes: Pre-defined bounding boxes [[x1,y1,x2,y2], ...].
                        If provided, skips auto-detection and uses these regions.
        text: Text(s) for computing text-image relevance scores per detection.

    Returns:
        dict with:
            - boundingBoxes: [[x1,y1,x2,y2], ...] for each detected item
            - tags: [[label, confidence], ...] per item (when withTag=True)
            - embeddings: [[float, ...], ...] per item (when withEmbedding=True)
            - textRelevanceScores: [float, ...] per item (when text is provided)
    """
    payload = {"image": image_url, "withTag": with_tag, "withEmbedding": with_embedding}
    if bounding_boxes is not None:
        payload["boundingBoxes"] = bounding_boxes
    if text is not None:
        payload["text"] = text
    resp = client.post("/model/image-embedding", json=payload)
    resp.raise_for_status()
    return resp.json()["data"]


# ── Display helpers ──────────────────────────────────────────────────────

def load_image(url: str) -> Image.Image:
    """Download an image from URL."""
    r = httpx.get(url, timeout=10, follow_redirects=True)
    return Image.open(BytesIO(r.content))


def show_query_and_results(query_img_url: str, results: list[dict],
                          query_title: str = "Query", max_results: int = 4,
                          bbox: list[int] | None = None):
    """Display query image alongside top matching products."""
    n = min(len(results), max_results)
    fig, axes = plt.subplots(1, n + 1, figsize=(4 * (n + 1), 5))

    # Query image
    try:
        qimg = load_image(query_img_url)
        if bbox:
            draw = ImageDraw.Draw(qimg)
            draw.rectangle(bbox, outline="red", width=3)
        axes[0].imshow(qimg)
    except Exception:
        axes[0].text(0.5, 0.5, "N/A", ha="center", va="center")
    axes[0].set_title(f"QUERY\n{query_title}", fontsize=10, fontweight="bold")
    axes[0].axis("off")

    # Result images
    for i, p in enumerate(results[:n]):
        ax = axes[i + 1]
        try:
            ax.imshow(load_image(p.get("imageUrl", "")))
        except Exception:
            ax.text(0.5, 0.5, "N/A", ha="center", va="center")
        cat = p.get("category") or p.get("productTag") or ""
        price = p.get("price")
        score = p.get("matchScore")
        label = f"#{i+1}"
        if cat:
            label += f" | {cat}"
        if price:
            label += f" | ${price:.2f}"
        if score:
            label += f"\nscore: {score:.3f}"
        ax.set_title(label, fontsize=8)
        ax.axis("off")

    plt.tight_layout()
    plt.show()


def show_detections(image_url: str, detections: list[dict], title: str = "Detected Fashion Items"):
    """Draw bounding boxes with labels on an image for each detection."""
    img = load_image(image_url)
    draw = ImageDraw.Draw(img)

    colors = ["#FF4444", "#44AA44", "#4444FF", "#FF8800", "#AA44AA",
              "#00AAAA", "#FFAA00", "#FF44AA", "#44AAFF", "#AAAA00"]

    for i, det in enumerate(detections):
        bbox = det["boundingBox"]
        class_id = det["classId"]
        score = det["score"]
        label = FASHION_CLASSES.get(class_id, f"Class {class_id}")
        color = colors[class_id % len(colors)]

        # Draw bbox
        draw.rectangle(bbox, outline=color, width=3)
        # Draw label background + text
        text_str = f"{label} {score:.2f}"
        draw.rectangle([bbox[0], bbox[1] - 16, bbox[0] + len(text_str) * 7, bbox[1]], fill=color)
        draw.text((bbox[0] + 2, bbox[1] - 14), text_str, fill="white")

    fig, ax = plt.subplots(1, 1, figsize=(8, 8))
    ax.imshow(img)
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.axis("off")
    plt.tight_layout()
    plt.show()

---
## Task 1: Get a Query Image

First, let's use APIClaw's text search to find a fashion product. We'll use its image as the query for visual similarity search.

In [ ]:
# Find a product to use as our query image
seed_products = text_search("women red leather handbag", top_k=3)

query_product = seed_products[0]
query_image_url = query_product.get("imageUrl", "")

print(f"Query product: {query_product.get('title', '')}")
print(f"Brand: {query_product.get('brand', '-')} | Price: ${query_product.get('price', 0):.2f}")
print(f"Image URL: {query_image_url[:80]}...")

# Display
try:
    plt.figure(figsize=(4, 4))
    plt.imshow(load_image(query_image_url))
    plt.title(f"{query_product.get('title', '')[:50]}")
    plt.axis("off")
    plt.show()
except Exception as e:
    print(f"Could not display image: {e}")

---
## Task 2: Basic Image Search

Pass the product image URL to the image search endpoint and find visually similar products across the entire catalog.

In [ ]:
# Search for visually similar products
result = image_search(query_image_url, limit=8)

print(f"Query image: {result.get('imageUrl', '')[:60]}...")
print(f"Relevance score: {result.get('relevanceScore')}")
print(f"Results: {result.get('total', 0)} products")

if result.get("bbox"):
    print(f"Detected bbox: {result['bbox']}")

products = result.get("products", [])
print(f"\nTop matches:")
for i, p in enumerate(products[:5]):
    print(f"  {i+1}. {p.get('category', '-'):>12} | "
          f"score: {p.get('matchScore', 0):.3f} | "
          f"${p.get('price', 0):.2f}")

In [ ]:
# Visualize query vs top matches
show_query_and_results(query_image_url, products, query_title="Red Handbag")

---
## Task 3: Fashion Item Detection

The **image-detection** endpoint (`POST /model/image-detection`) automatically locates fashion items in any photo — no manual bounding boxes needed. It returns:

- **Bounding box** `[x1, y1, x2, y2]` for each detected item
- **Category class ID** (0=Bag, 1=Cap, 2=Down-Clothing, 3=Glasses, 4=Jewelry, 5=Others, 6=Shoes, 7=Sock, 8=Up-Clothing, 9=Watch)
- **Confidence score** (0-1)
- **Area ratio** and **center distance ratio** for spatial analysis

This is the first step in any "Shop the Look" pipeline: detect what's in the photo before searching for each item.

#### Request schema

```json
{
    "image": "https://...",     // HTTPS URL (required)
    "topK": 10,                 // max detections (required, 1-50)
    "classes": [0, 6],          // filter by category IDs (optional, null = all)
    "returnImage": false        // return annotated image with boxes drawn (optional)
}
```

In [ ]:
# Detect all fashion items in our query image
detection_result = detect_fashion_items(query_image_url, top_k=10)

detections = detection_result["detections"]
print(f"Detected {len(detections)} fashion items:\n")
for i, det in enumerate(detections):
    class_name = FASHION_CLASSES.get(det["classId"], f"Unknown({det['classId']})")
    bbox = det["boundingBox"]
    print(f"  {i+1}. {class_name:>14} | confidence: {det['score']:.3f} | "
          f"bbox: [{bbox[0]:.0f}, {bbox[1]:.0f}, {bbox[2]:.0f}, {bbox[3]:.0f}] | "
          f"area: {det['areaRatio']:.1%}")

# Visualize detections with bounding boxes
show_detections(query_image_url, detections)

In [ ]:
# Filter detection to specific categories — e.g. only Bags (0) and Shoes (6)
filtered = detect_fashion_items(query_image_url, top_k=5, classes=[0, 6])

print(f"Filtered detections (Bags + Shoes only): {len(filtered['detections'])} items")
for det in filtered["detections"]:
    class_name = FASHION_CLASSES.get(det["classId"], "?")
    print(f"  {class_name} | confidence: {det['score']:.3f}")

show_detections(query_image_url, filtered["detections"], title="Filtered: Bags + Shoes Only")

---
## Task 4: Image Embedding — Detect + Embed in One Call

The **image-embedding** endpoint (`POST /model/image-embedding`) is the all-in-one workhorse: it detects fashion items, classifies them, and generates feature embedding vectors — all in a single API call.

This is what you'd use to build a **custom visual search index** or compute **similarity between images** programmatically.

#### Request schema

```json
{
    "image": "https://...",              // HTTPS URL (required)
    "withTag": true,                     // include category tags (default: false)
    "withEmbedding": true,               // include feature vectors (default: true)
    "boundingBoxes": [[x1,y1,x2,y2]],   // skip auto-detection, use these regions (optional)
    "text": ["red dress", "blue jeans"]  // compute text-image relevance per item (optional)
}
```

#### Response structure

```json
{
    "boundingBoxes": [[x1,y1,x2,y2], ...],       // one per detected item
    "tags": [["Bag", 0.97], ["Shoes", 0.92]],     // [label, confidence] per item
    "embeddings": [[0.12, -0.34, ...], ...],       // feature vector per item
    "textRelevanceScores": [0.85, 0.23, ...]       // text-image score per item
}
```

**Key insight:** You can pass bounding boxes from the detection endpoint (Task 3) into this endpoint to get embeddings for specific detected regions — or let it auto-detect and embed in one shot.

In [ ]:
# Auto-detect + embed + tag — all in one call
emb_result = get_image_embedding(query_image_url, with_tag=True, with_embedding=True)

bboxes = emb_result["boundingBoxes"]
tags = emb_result.get("tags", [])
embeddings = emb_result.get("embeddings", [])

print(f"Detected {len(bboxes)} items with embeddings:\n")
for i, (bbox, tag, emb) in enumerate(zip(bboxes, tags, embeddings)):
    label, confidence = tag[0], tag[1]
    print(f"  {i+1}. {label:>14} (conf: {confidence:.3f})")
    print(f"     bbox: [{bbox[0]}, {bbox[1]}, {bbox[2]}, {bbox[3]}]")
    print(f"     embedding: [{emb[0]:.4f}, {emb[1]:.4f}, {emb[2]:.4f}, ...] (dim={len(emb)})")
    print()

In [ ]:
# Pass pre-defined bounding boxes from detection (Task 3) to get embeddings
# This gives you fine-grained control: detect first, pick regions, then embed
det_bboxes = [[int(c) for c in det["boundingBox"]] for det in detections[:3]]

emb_from_det = get_image_embedding(
    query_image_url,
    with_tag=True,
    with_embedding=True,
    bounding_boxes=det_bboxes,  # use detection results instead of auto-detect
)

print("Embeddings from detection bounding boxes:\n")
for i, (bbox, tag, emb) in enumerate(zip(
    emb_from_det["boundingBoxes"],
    emb_from_det.get("tags", []),
    emb_from_det.get("embeddings", []),
)):
    label, confidence = tag[0], tag[1]
    print(f"  {i+1}. {label} (conf: {confidence:.3f}) → embedding dim={len(emb)}")

In [ ]:
# Use text relevance scoring to rank detected items against a description
emb_with_text = get_image_embedding(
    query_image_url,
    with_tag=True,
    with_embedding=False,  # skip embeddings — just want tags + text scores
    text=["red leather handbag", "black high heels"],
)

print("Text-image relevance scores per detected item:\n")
print(f"  Text queries: ['red leather handbag', 'black high heels']\n")
for i, (tag, score) in enumerate(zip(
    emb_with_text.get("tags", []),
    emb_with_text.get("textRelevanceScores", []),
)):
    label, confidence = tag[0], tag[1]
    print(f"  Item {i+1}: {label} → text relevance: {score:.4f}")

---
## Task 5: Manual Bounding Box Cropping

When you already know which region of an image contains the item you want (e.g., from Task 3's detection results), use the `bbox` parameter on the image search endpoint to focus on that specific area. The bounding box is specified as `[x1, y1, x2, y2]` in pixels.

This is useful when you want to manually define regions — for automated workflows, use the detection endpoint from Task 3 instead.

In [ ]:
# Get a full-body fashion image (outfit photo)
outfit_products = text_search("women complete outfit dress shoes", top_k=3)
outfit_url = outfit_products[0].get("imageUrl", query_image_url)

# Load and measure the image to set a reasonable bbox
try:
    outfit_img = load_image(outfit_url)
    w, h = outfit_img.size
    print(f"Image size: {w}x{h}")

    # Focus on the upper half (e.g., top/jacket area)
    upper_bbox = [0, 0, w, h // 2]
    # Focus on the lower half (e.g., shoes/pants area)
    lower_bbox = [0, h // 2, w, h]

    print(f"Upper region bbox: {upper_bbox}")
    print(f"Lower region bbox: {lower_bbox}")
except Exception as e:
    print(f"Using fallback bbox: {e}")
    upper_bbox = [0, 0, 300, 300]
    lower_bbox = [0, 300, 300, 600]

In [ ]:
# Search using the upper region
upper_result = image_search(outfit_url, limit=4, bbox=upper_bbox)
upper_products = upper_result.get("products", [])

print(f"Upper region: {len(upper_products)} matches")
show_query_and_results(outfit_url, upper_products,
                       query_title="Upper Region", bbox=upper_bbox)

In [ ]:
# Search using the lower region
lower_result = image_search(outfit_url, limit=4, bbox=lower_bbox)
lower_products = lower_result.get("products", [])

print(f"Lower region: {len(lower_products)} matches")
show_query_and_results(outfit_url, lower_products,
                       query_title="Lower Region", bbox=lower_bbox)

---
## Task 6: Text-Guided Image Search

Add an `imageDescription` to guide the visual search. This helps when:
- The image contains multiple items and you want a specific one
- You want to bias results toward a particular style or color
- The image is ambiguous (e.g., a bag could be a tote or clutch)

In [ ]:
# Same image, different text guidance
descriptions = [
    None,  # baseline: image only
    "red leather crossbody bag",
    "designer evening clutch",
]

for desc in descriptions:
    kwargs = {}
    if desc:
        kwargs["imageDescription"] = desc

    result = image_search(query_image_url, limit=4, **kwargs)
    products = result.get("products", [])

    label = desc or "(no description)"
    print(f'\nDescription: "{label}"')
    for i, p in enumerate(products[:3]):
        print(f"  {i+1}. {p.get('category', '-'):>10} | "
              f"score: {p.get('matchScore', 0):.3f} | "
              f"${p.get('price', 0):.2f}")

In [ ]:
# Visualize how text guidance changes results
fig, axes = plt.subplots(len(descriptions), 5, figsize=(18, 4 * len(descriptions)))

for row, desc in enumerate(descriptions):
    kwargs = {}
    if desc:
        kwargs["imageDescription"] = desc
    result = image_search(query_image_url, limit=4, **kwargs)
    products = result.get("products", [])

    # Query
    try:
        axes[row, 0].imshow(load_image(query_image_url))
    except Exception:
        pass
    axes[row, 0].set_title(f"Query\n\"{desc or 'image only'}\"", fontsize=9, fontweight="bold")
    axes[row, 0].axis("off")

    # Results
    for j, p in enumerate(products[:4]):
        ax = axes[row, j + 1]
        try:
            ax.imshow(load_image(p.get("imageUrl", "")))
        except Exception:
            pass
        score = p.get("matchScore", 0)
        cat = p.get("category", "")
        ax.set_title(f"{cat}\nscore: {score:.3f}", fontsize=8)
        ax.axis("off")

plt.suptitle("Effect of Text Guidance on Image Search", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

---
## Task 7: Filtered Image Search

Combine visual search with structured filters: brands, price range, and retailer domains.

In [ ]:
# Image search with price filter: find affordable alternatives
affordable = image_search(
    query_image_url,
    limit=5,
    priceMin=20,
    priceMax=100,
)

print(f"Affordable alternatives ($20-$100): {affordable.get('total', 0)} results")
affordable_products = affordable.get("products", [])
for i, p in enumerate(affordable_products):
    print(f"  {i+1}. ${p.get('price', 0):>6.2f} | {p.get('category', '-')} | score: {p.get('matchScore', 0):.3f}")

show_query_and_results(query_image_url, affordable_products, query_title="Find Affordable Dupes")

In [ ]:
# Image search with brand filter
branded = image_search(
    query_image_url,
    limit=5,
    brands=["Coach", "Michael Kors", "Kate Spade"],
)

branded_products = branded.get("products", [])
print(f"Brand-filtered results: {branded.get('total', 0)}")
for p in branded_products:
    print(f"  ${p.get('price', 0):>6.2f} | score: {p.get('matchScore', 0):.3f} | {p.get('category', '-')}")

show_query_and_results(query_image_url, branded_products, query_title="Branded Matches")

In [ ]:
# Image search with retailer filter
retailer_results = image_search(
    query_image_url,
    limit=5,
    sites=["nordstrom.com", "farfetch.com"],
)

retailer_products = retailer_results.get("products", [])
print(f"From Nordstrom + Farfetch: {retailer_results.get('total', 0)} results")
for p in retailer_products:
    print(f"  ${p.get('price', 0):>6.2f} | score: {p.get('matchScore', 0):.3f} | {p.get('category', '-')}")

---
## Task 8: "Shop the Look" Pipeline — Detect → Search

The complete production pipeline for visual product discovery from a street-style or outfit photo:

1. **Detect** — use `/model/image-detection` to find all fashion items with bounding boxes
2. **Classify** — the detection response includes category labels (Bag, Shoes, Up-Clothing, etc.)
3. **Search** — for each detected item, use `/model/fashion-image-search` with the detected bbox to find matching products

This replaces the manual "split image into thirds" approach with intelligent, model-driven item localization.

In [ ]:
# Step 1: Get an outfit image
outfit_items = text_search("women street style outfit", top_k=3)
look_url = outfit_items[0].get("imageUrl", query_image_url)

# Step 2: Detect all fashion items automatically
look_detections = detect_fashion_items(look_url, top_k=10)["detections"]

print(f"Detected {len(look_detections)} fashion items in the outfit photo:\n")
for i, det in enumerate(look_detections):
    class_name = FASHION_CLASSES.get(det["classId"], "Unknown")
    print(f"  {i+1}. {class_name:>14} | confidence: {det['score']:.3f} | "
          f"area: {det['areaRatio']:.1%}")

# Visualize all detections
show_detections(look_url, look_detections, title="Step 1: Detect Fashion Items")

# Step 3: Search for matching products for each detected item
print("\n" + "=" * 60)
print("Step 2: Searching matching products for each detected item...")
print("=" * 60)

shop_the_look = {}
for det in look_detections:
    class_name = FASHION_CLASSES.get(det["classId"], "Unknown")
    bbox = [int(c) for c in det["boundingBox"]]

    # Use the detected bounding box + category as text guidance
    result = image_search(
        look_url,
        limit=3,
        bbox=bbox,
        imageDescription=class_name.lower(),
    )
    products = result.get("products", [])
    shop_the_look[f"{class_name} (conf: {det['score']:.2f})"] = {
        "bbox": bbox,
        "products": products,
    }

    print(f"\n  {class_name} (conf: {det['score']:.2f}): {len(products)} matches")
    for p in products:
        print(f"    {p.get('category', '-'):>12} | "
              f"${p.get('price', 0):.2f} | score: {p.get('matchScore', 0):.3f}")

In [ ]:
# Visualize the "Shop the Look" results — one row per detected item
items = list(shop_the_look.items())
n_items = len(items)

if n_items > 0:
    fig, axes = plt.subplots(n_items, 4, figsize=(16, 4 * n_items))
    if n_items == 1:
        axes = [axes]

    for row, (item_label, item_data) in enumerate(items):
        bbox = item_data["bbox"]
        products = item_data["products"]

        # Query region with bbox overlay
        try:
            rimg = load_image(look_url)
            draw = ImageDraw.Draw(rimg)
            draw.rectangle(bbox, outline="red", width=3)
            axes[row][0].imshow(rimg)
        except Exception:
            axes[row][0].text(0.5, 0.5, "N/A", ha="center", va="center")
        axes[row][0].set_title(f"{item_label}\nbbox: {bbox}", fontsize=9, fontweight="bold")
        axes[row][0].axis("off")

        # Matched products
        for j in range(3):
            ax = axes[row][j + 1]
            if j < len(products):
                p = products[j]
                try:
                    ax.imshow(load_image(p.get("imageUrl", "")))
                except Exception:
                    pass
                price = p.get("price", 0)
                score = p.get("matchScore", 0)
                ax.set_title(f"${price:.2f} | score: {score:.3f}", fontsize=8)
            else:
                ax.text(0.5, 0.5, "No match", ha="center", va="center")
            ax.axis("off")

    plt.suptitle("Shop the Look — Detect → Search", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()

---
## Summary

This notebook demonstrated how to use APIClaw's fashion vision APIs for visual product discovery — from basic image search to a full detect → embed → search pipeline.

### APIs Used

| Endpoint | Purpose | Cost |
|----------|---------|------|
| `POST /model/fashion-image-search` | Visual similarity product search | 1 credit |
| `POST /model/fashion-product-search` | Text search (for demo images) | 1 credit |
| `POST /model/image-detection` | Detect fashion items → bounding boxes + category + confidence | 1 credit |
| `POST /model/image-embedding` | Detect + classify + generate feature embeddings (all-in-one) | 1 credit |

### Detection Categories

| Class ID | Category | Class ID | Category |
|----------|----------|----------|----------|
| 0 | Bag | 5 | Others |
| 1 | Cap | 6 | Shoes |
| 2 | Down-Clothing | 7 | Sock |
| 3 | Glasses | 8 | Up-Clothing |
| 4 | Jewelry | 9 | Watch |

### Key Parameters

| Endpoint | Parameter | Description |
|----------|-----------|-------------|
| image-search | `imageUrl` | HTTPS URL of the query image |
| image-search | `bbox` | Crop region `[x1, y1, x2, y2]` in pixels |
| image-search | `imageDescription` | Text to compose with image for guided search |
| image-search | `brands` / `sites` / `priceMin` / `priceMax` | Structured filters |
| image-detection | `topK` | Max detections (1-50) |
| image-detection | `classes` | Filter by category IDs (e.g. `[0, 6]` for Bag + Shoes) |
| image-embedding | `withTag` / `withEmbedding` | Toggle tags and embedding vectors |
| image-embedding | `boundingBoxes` | Pre-defined regions (skip auto-detection) |
| image-embedding | `text` | Compute text-image relevance scores per detection |

### Production Patterns

- **"Find Similar"**: Basic image search → visually similar products
- **"Shop the Look"**: Detect items → use detected bboxes → per-item product search
- **"Custom Index"**: Detect + embed → store vectors → build your own similarity search
- **"Affordable Dupes"**: Image search + price filter → budget alternatives
- **"Brand Match"**: Image search + brand filter → find specific brand alternatives

### Also Available (not covered in this notebook)

| Endpoint | Description |
|----------|-------------|
| `POST /model/fashion-image-embedding` | SigLIP2 768-dim image embeddings (batch up to 8) |
| `POST /model/fashion-text-embedding` | SigLIP2 768-dim text embeddings (batch up to 32) |
| `POST /model/fashion-similarity` | Text-image cosine similarity scoring matrix |

These endpoints are useful for building custom reranking pipelines or cross-modal search indexes.

### Next Steps

- Try the [Fashion Product Search](https://colab.research.google.com/github/SerendipityOneInc/APIClaw-Notebook/blob/main/e-commerce-search/product_search.ipynb) notebook for text-based discovery
- Combine detection + SigLIP2 embeddings for custom reranking pipelines
- Sign up at [apiclaw.io](https://apiclaw.io) and explore our [credit packages](https://apiclaw.io/pricing) for production workloads